# Hospital Management Analytics
## Dataset analysis, data quality, statistical graphs, and dashboard preparation

This notebook analyzes the supplied `hospital_filtered_records.csv`. The file includes hospital records and derived fields such as `length_of_stay`, `billing_review`, and `invalid_stay`.

**Workflow:** load data → inspect quality → prepare fields → explore statistics → visualize distributions and relationships → save a cleaned copy.

> The supplied file is named `hospital_filtered_records.csv`; it is treated as the dataset supplied for this project, not assumed to be an untouched original source.

## 1. Import libraries and load dataset

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA_PATH = Path("../data/hospital_filtered_records.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("data/hospital_filtered_records.csv")

df_raw = pd.read_csv(DATA_PATH)
print("Dataset shape:", df_raw.shape)
display(df_raw.head())
display(pd.DataFrame({"column": df_raw.columns, "dtype": df_raw.dtypes.astype(str).values}))

## 2. Data overview and quality

In [ ]:
print("Rows:", f"{len(df_raw):,}")
print("Columns:", df_raw.shape[1])
print("\\nMissing values by column:")
display(df_raw.isna().sum().sort_values(ascending=False).to_frame("missing_count"))
print("\\nDuplicate rows:", int(df_raw.duplicated().sum()))
display(df_raw.describe(include="all").T)

## 3. Prepare an analysis copy

In [ ]:
df = df_raw.copy()
df.columns = [str(c).strip() for c in df.columns]

for col in ["age", "billing_amount", "room_number", "length_of_stay"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

for col in ["date_of_admission", "discharge_date"]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

for col in df.select_dtypes(include=["object"]).columns:
    df[col] = df[col].map(lambda x: x.strip() if isinstance(x, str) else x)

print("Analysis copy ready:", df.shape)
display(df.head())

## 4. Missing data visualization

In [ ]:
missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
missing_pct = missing_pct[missing_pct > 0]
if not missing_pct.empty:
    ax = missing_pct.sort_values().plot(kind="barh", figsize=(10, 6))
    ax.set_title("Missing Values by Column")
    ax.set_xlabel("Missing values (%)")
    ax.set_ylabel("Column")
    plt.tight_layout()
    plt.show()
else:
    print("No missing values found.")

## 5. Patient age distribution

In [ ]:
if "age" in df.columns:
    ages = df["age"].dropna()
    display(ages.describe().to_frame("age"))
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.hist(ages, bins=30, edgecolor="black")
    ax.set_title("Patient Age Distribution")
    ax.set_xlabel("Age")
    ax.set_ylabel("Number of records")
    plt.tight_layout()
    plt.show()
else:
    print("Age column not found.")

## 6. Categorical analysis

In [ ]:
for col in ["medical_condition", "admission_type", "gender", "test_results", "medication"]:
    if col in df.columns:
        counts = df[col].fillna("Unknown").astype(str).value_counts().head(20)
        display(counts.rename_axis(col).to_frame("record_count"))
        ax = counts.sort_values().plot(kind="barh", figsize=(9, max(3, len(counts) * 0.28)))
        ax.set_title(f"Records by {col.replace('_', ' ').title()}")
        ax.set_xlabel("Records")
        plt.tight_layout()
        plt.show()

## 7. Billing distribution and outliers

In [ ]:
if "billing_amount" in df.columns:
    bills = df["billing_amount"].dropna()
    display(bills.describe().to_frame("billing_amount"))
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(bills, bins=35, edgecolor="black")
    axes[0].set_title("Billing Amount Distribution")
    axes[0].set_xlabel("Billing amount")
    axes[0].set_ylabel("Frequency")
    axes[1].boxplot(bills, vert=True)
    axes[1].set_title("Billing Amount Boxplot")
    axes[1].set_ylabel("Billing amount")
    plt.tight_layout()
    plt.show()
else:
    print("Billing column not found.")

## 8. Billing by medical condition

In [ ]:
if {"medical_condition", "billing_amount"}.issubset(df.columns):
    grouped = df.groupby("medical_condition", dropna=False)["billing_amount"].agg(["count", "mean", "median", "std"])
    display(grouped.sort_values("mean", ascending=False))
    ax = grouped["mean"].sort_values().plot(kind="barh", figsize=(10, 5))
    ax.set_title("Average Billing by Medical Condition")
    ax.set_xlabel("Average billing")
    plt.tight_layout()
    plt.show()

## 9. Numeric correlation analysis

In [ ]:
numeric = df.select_dtypes(include="number")
numeric = numeric.loc[:, numeric.nunique(dropna=True) > 1]
if numeric.shape[1] >= 2:
    corr = numeric.corr()
    display(corr)
    fig, ax = plt.subplots(figsize=(max(7, len(corr.columns) * 0.8), max(5, len(corr.columns) * 0.65)))
    im = ax.imshow(corr, vmin=-1, vmax=1, aspect="auto")
    ax.set_xticks(range(len(corr.columns)), corr.columns, rotation=70, ha="right")
    ax.set_yticks(range(len(corr.columns)), corr.columns)
    ax.set_title("Pearson Correlation Heatmap")
    fig.colorbar(im, ax=ax, label="Correlation")
    plt.tight_layout()
    plt.show()
else:
    print("At least two varying numeric columns are needed for correlation.")

## 10. Admission trends and length of stay

In [ ]:
if "date_of_admission" in df.columns:
    date_df = df.dropna(subset=["date_of_admission"]).copy()
    if not date_df.empty:
        date_df["admission_month"] = date_df["date_of_admission"].dt.to_period("M").astype(str)
        monthly = date_df.groupby("admission_month").size()
        display(monthly.to_frame("admissions"))
        ax = monthly.plot(figsize=(11, 4), marker="o")
        ax.set_title("Monthly Admissions")
        ax.set_xlabel("Admission month")
        ax.set_ylabel("Number of records")
        plt.xticks(rotation=60)
        plt.tight_layout()
        plt.show()

if "length_of_stay" in df.columns:
    los = df["length_of_stay"].dropna()
    if not los.empty:
        display(los.describe().to_frame("length_of_stay_days"))
        ax = los.plot(kind="hist", bins=30, figsize=(9, 4), edgecolor="black")
        ax.set_title("Length of Stay Distribution")
        ax.set_xlabel("Length of stay (days)")
        plt.tight_layout()
        plt.show()
elif {"date_of_admission", "discharge_date"}.issubset(df.columns):
    df["calculated_length_of_stay"] = (df["discharge_date"] - df["date_of_admission"]).dt.total_seconds() / 86400
    display(df["calculated_length_of_stay"].describe())

## 11. Save cleaned dataset for dashboard use

In [ ]:
CLEAN_PATH = Path("../data/hospital_cleaned.csv")
if not CLEAN_PATH.parent.exists():
    CLEAN_PATH = Path("data/hospital_cleaned.csv")
CLEAN_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(CLEAN_PATH, index=False)
print(f"Cleaned analysis dataset saved to: {CLEAN_PATH.resolve()}")

## 12. Findings and interpretation

After running the notebook, record findings based on the output:
- Which medical conditions are most frequent?
- What do the age and billing distributions show?
- Which columns have the most missing values?
- Are any numeric variables associated?
- What patterns appear in monthly admissions or length of stay?

Correlation does not prove causation. Verify unusual values and missing data before drawing conclusions.